In [ ]:
!pip install marker-pdf==0.3.10
!pip install texify==0.1.10

In [ ]:
!pip install "surya-ocr==0.6.13" "transformers==4.41.0" "tabled-pdf==0.1.4"

In [ ]:
import os
import shutil
import json
from pathlib import Path
from google.colab import drive
from marker.convert import convert_single_pdf
from marker.models import load_all_models
from marker.output import save_markdown

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Paths
DRIVE_BASE = Path("/content/drive/MyDrive/Papers Processing")
INPUT_DIR = DRIVE_BASE / "papers"
MARKDOWN_DIR = DRIVE_BASE / "extracted_markdown"
DONE_DIR = DRIVE_BASE / "papers_done"

for folder in [MARKDOWN_DIR, DONE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

def process_papers():
    # 3. Load Models (marker will use your Colab GPU automatically if selected)
    print("Loading AI Models for Math, Tables, and Layout...")
    models = load_all_models()
    print("Models loaded.")

    # 4. Get PDFs
    pdf_files = sorted([f for f in INPUT_DIR.iterdir() if f.suffix.lower() == ".pdf"])

    if not pdf_files:
        print("No PDFs found to process.")
        return

    print(f"Starting pipeline for {len(pdf_files)} papers...")

    for i, pdf_path in enumerate(pdf_files, 1):
        try:
            # 5. Handle the ID safely
            # Instead of .stem, we take the full name and remove '.pdf' ex: "1706.03762.pdf" -> "1706.03762"
            raw_id = pdf_path.name.replace(".pdf", "").replace(".PDF", "")

            # For filenames, replace '.' with '_' to avoid extension confusion
            safe_id = raw_id.replace(".", "_")

            # 6. Create a dedicated folder for THIS paper
            # Result: /extracted_markdown/1706.03762/
            paper_folder = MARKDOWN_DIR / raw_id
            paper_folder.mkdir(parents=True, exist_ok=True)

            print(f"\n[{i}/{len(pdf_files)}] Extracting: {raw_id}")

            # 7. The Optimal Extraction Call
            full_text, images, metadata = convert_single_pdf(
                str(pdf_path),
                models,
                langs=["en"]
            )

            # 8. Save using the professional helper
            # save_markdown(str(paper_folder), safe_id, full_text, images, metadata)

            # --- MANUAL FLAT-SAVE LOGIC ---
            # Save the .md file (e.g., 1409_3215.md)
            md_file_path = paper_folder / f"{safe_id}.md"
            with open(md_file_path, "w", encoding="utf-8") as f:
                f.write(full_text)

            # Save the metadata as a JSON file
            meta_file_path = paper_folder / f"{safe_id}_meta.json"
            with open(meta_file_path, "w", encoding="utf-8") as f:
                json.dump(metadata, f, indent=4)

            # Save the images directly inside the 1409.3215 folder
            if images:
                for img_name, img_obj in images.items():
                    # img_name usually looks like "1_img_0.png"
                    img_obj.save(paper_folder / img_name)

            # 9. Move to 'done' only after save is successful
            shutil.move(str(pdf_path), str(DONE_DIR / pdf_path.name))
            print(f"Success. Moved {pdf_path.name} to papers_done.")

        except Exception as e:
            print(f"Error on {pdf_path.name}: {e}")
            continue

if __name__ == "__main__":
    process_papers()
    print("\nAll papers processed. Your high-quality RAG dataset is ready.")

In [ ]:
import os
import shutil
from pathlib import Path
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Setup Paths
DRIVE_BASE = Path("/content/drive/MyDrive/Papers Processing")
INPUT_DIR = DRIVE_BASE / "papers"
DONE_DIR = DRIVE_BASE / "papers_done"
MARKDOWN_DIR = DRIVE_BASE / "extracted_markdown"

In [ ]:
def count_items(folder_path, is_pdf=True):
    if not folder_path.exists():
        return 0
    if is_pdf:
        return len([f for f in folder_path.iterdir() if f.suffix.lower() == ".pdf"])
    else:
        # Count only directories in the extraction folder
        return len([f for f in folder_path.iterdir() if f.is_dir()])

print(f"📊 Full Pipeline Status:")
print(f"--------------------------")
print(f"📂 'papers' (To Process):         {count_items(INPUT_DIR, is_pdf=True)}")
print(f"✅ 'papers_done' (Moved):         {count_items(DONE_DIR, is_pdf=True)}")
print(f"📁 'extracted_markdown' (Folders): {count_items(MARKDOWN_DIR, is_pdf=False)}")

# Validation Logic
done_count = count_items(DONE_DIR, is_pdf=True)
folder_count = count_items(MARKDOWN_DIR, is_pdf=False)

if done_count == folder_count:
    print(f"\n✨ Sync Status: Perfect. Every moved PDF has a corresponding folder.")
else:
    print(f"\n⚠️ Sync Status: Mismatch. {abs(done_count - folder_count)} items are out of sync.")

In [ ]:
def reset_papers():
    # Check if the folder exists
    if not DONE_DIR.exists():
        print(f"The folder {DONE_DIR} does not exist.")
        return

    # Ensure the target folder exists
    INPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Gather all PDFs in papers_done
    files_to_move = [f for f in DONE_DIR.iterdir() if f.suffix.lower() == ".pdf"]

    if not files_to_move:
        print("No PDF files found in 'papers_done' to move.")
        return

    print(f"Moving {len(files_to_move)} papers back to 'papers'...")

    for pdf_path in files_to_move:
        try:
            # Move the file
            dest_path = INPUT_DIR / pdf_path.name
            shutil.move(str(pdf_path), str(dest_path))
            print(f"Moved: {pdf_path.name}")
        except Exception as e:
            print(f"Error moving {pdf_path.name}: {e}")

    print("\nReset complete. You can now run your updated extraction script.")

reset_papers()

In [ ]:
def count_items(folder_path, is_pdf=True):
    if not folder_path.exists():
        return 0
    if is_pdf:
        return len([f for f in folder_path.iterdir() if f.suffix.lower() == ".pdf"])
    else:
        # Count only directories in the extraction folder
        return len([f for f in folder_path.iterdir() if f.is_dir()])

print(f"📊 Full Pipeline Status:")
print(f"--------------------------")
print(f"📂 'papers' (To Process):         {count_items(INPUT_DIR, is_pdf=True)}")
print(f"✅ 'papers_done' (Moved):         {count_items(DONE_DIR, is_pdf=True)}")
print(f"📁 'extracted_markdown' (Folders): {count_items(MARKDOWN_DIR, is_pdf=False)}")

# Validation Logic
done_count = count_items(DONE_DIR, is_pdf=True)
folder_count = count_items(MARKDOWN_DIR, is_pdf=False)

if done_count == folder_count:
    print(f"\n✨ Sync Status: Perfect. Every moved PDF has a corresponding folder.")
else:
    print(f"\n⚠️ Sync Status: Mismatch. {abs(done_count - folder_count)} items are out of sync.")

In [ ]:
# WARNING: This deletes the entire 'papers_done' & 'extracted_markdown' folder to start fresh
shutil.rmtree('/content/drive/MyDrive/Papers Processing/papers_done', ignore_errors=True)
shutil.rmtree('/content/drive/MyDrive/Papers Processing/extracted_markdown', ignore_errors=True)